In [1]:
import numpy as np
import time

print("--- Q1: Vectorize x^2 + 2x + 1 for 1 million values ---")
x = np.arange(1_000_000)
start_time = time.time()
y_vectorized = x**2 + 2*x + 1
print(f"Vectorized calculation took {time.time() - start_time:.4f} seconds")

print("\n--- Q2: Numba function for Sum of Squares ---")
try:
    from numba import njit
    
    @njit
    def sum_squares_numba(arr):
        total = 0
        for i in range(len(arr)):
            total += arr[i]**2
        return total

    # First run (compilation)
    sum_squares_numba(x)
    
    # Second run (actual execution)
    start_time = time.time()
    result = sum_squares_numba(x)
    print(f"Numba sum of squares took {time.time() - start_time:.4f} seconds")
    print(f"Result: {result}")
except ImportError:
    print("Numba is not installed in this environment. (The syntax above is correct).")

--- Q1: Vectorize x^2 + 2x + 1 for 1 million values ---
Vectorized calculation took 0.0380 seconds

--- Q2: Numba function for Sum of Squares ---
Numba is not installed in this environment. (The syntax above is correct).


In [2]:
import numpy as np
import time

print("--- Q3: Parallel Bootstrap using joblib ---")
try:
    from joblib import Parallel, delayed
    
    data = np.random.normal(50, 10, 1000)
    
    def bootstrap_mean(data):
        sample = np.random.choice(data, size=len(data), replace=True)
        return np.mean(sample)

    start_time = time.time()
    # Run 1000 bootstraps in parallel across CPU cores
    results = Parallel(n_jobs=-1)(delayed(bootstrap_mean)(data) for _ in range(1000))
    print(f"Parallel bootstrap of 1000 samples took {time.time() - start_time:.4f} seconds")
    print(f"Mean of bootstrap means: {np.mean(results):.4f}")
except ImportError:
    print("joblib is not installed. (The syntax above is correct).")

print("\n--- Q4: Dask Array for Chunked Mean ---")
try:
    import dask.array as da
    
    # Create a large dask array (10000x10000) chunked into blocks
    # This doesn't load into memory all at once
    large_array = da.random.normal(10, 2, size=(10000, 10000), chunks=(1000, 1000))
    
    start_time = time.time()
    mean_val = large_array.mean().compute() # Triggers the actual calculation
    print(f"Dask calculated the mean of 100M elements in {time.time() - start_time:.4f} seconds")
    print(f"Calculated mean: {mean_val:.4f}")
except ImportError:
    print("Dask is not installed. (The syntax above is correct).")

--- Q3: Parallel Bootstrap using joblib ---
joblib is not installed. (The syntax above is correct).

--- Q4: Dask Array for Chunked Mean ---
Dask is not installed. (The syntax above is correct).


## LAB 13: Questions & Answers





**Q1. Vectorize the calculation x^2 + 2x + 1 for one million values.**
*Answer:* The calculation was fully vectorized using `x**2 + 2*x + 1`. This runs entirely in compiled C code over the memory buffer, avoiding Python loops, which makes it incredibly fast for a million elements.

**Q2. Write a Numba function that sums the squares.**
*Answer:* Using the `@njit` decorator from the `numba` library, a standard Python `for` loop was written to sum the squares. Numba compiles this loop to machine code at runtime, making it nearly as fast as (and sometimes faster than) NumPy's built-in functions.

**Q3. Bootstrap a sample mean in parallel using joblib.**
*Answer:* Using `Parallel(n_jobs=-1)` and `delayed`, the bootstrap function was executed across all available CPU cores simultaneously. This drastically reduces the time it takes to run 1,000 bootstraps compared to running them sequentially in a single loop.

**Q4. Use Dask Array to calculate the mean of a large chunked array.**
*Answer:* A 10,000 x 10,000 array was created using `dask.array`. Dask splits the array into smaller "chunks" so it can calculate the mean without loading the entire 100 million-element array into RAM at once, preventing memory overflow.

**Q5. Explain why benchmark timing should exclude Numba's first compilation call.**
*Answer:* Numba uses Just-In-Time (JIT) compilation. The *first* time a Numba function is called, it compiles the Python code into optimized machine code, which takes a significant amount of time (often milliseconds to seconds). Subsequent calls use the cached compiled code and run extremely fast. Excluding the first call ensures you are benchmarking the actual execution speed of the code, not the compilation overhead.